# Pré-processamento
* Separar em 9 grupos, cada estação de meteriológica, e um com o dado de todas
* Realizando um agrupamento por ano e estação do ano (verão, outono, primavera e inverno) o calculo de média e desvio padrão do valor de temperatura.
    * Obs: algumas estação começam a ter dados em anos diferentes
* Ter como entrada o arquivo em `/data_set/data_set_completo.csv` e como saída criar um diretório `/data_set/segunda_visualizacao/` e salvar cada um dos 9 grupos com o nome do grupo e em um .csv.

In [7]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import os
import numpy as np

In [ ]:
# 1. Configuração dos caminhos
caminho_entrada = "./data_set/data_set_completo.csv"
diretorio_saida = "./data_set/segunda_visualizacao/"
os.makedirs(diretorio_saida, exist_ok=True)

# Carregar o dataset
df = pd.read_csv(caminho_entrada)

# Converter datas e extrair Ano e Estação do Ano
df["data"] = pd.to_datetime(df["data"])
df["Ano"] = df["data"].dt.year


def obter_estacao_sul(dt):
    mes = dt.month
    dia = dt.day
    if (mes == 12 and dia >= 21) or (mes in [1, 2]) or (mes == 3 and dia < 20):
        return "Verão"
    elif (mes == 3 and dia >= 20) or (mes in [4, 5]) or (mes == 6 and dia < 21):
        return "Outono"
    elif (mes == 6 and dia >= 21) or (mes in [7, 8]) or (mes == 9 and dia < 22):
        return "Inverno"
    else:
        return "Primavera"


df["Estacao_Ano"] = df["data"].apply(obter_estacao_sul)

# --- 2. Preparação dos Vetores do Vento ---
# Convertemos a direção de graus para radianos
radianos = np.deg2rad(df["dir_vento"])

# Decomposição vetorial ponderada pela velocidade do vento
df["vento_x"] = df["vel_vento"] * np.cos(radianos)
df["vento_y"] = df["vel_vento"] * np.sin(radianos)


# --- 3. Função de Agrupamento Customizada ---
def processar_grupo(dataframe, incluir_coordenadas=False):
    # Dicionário de agregações base (Temperatura e componentes do vento)
    agregacoes = {
        "temp": ["mean", "std"],
        "vento_x": ["mean"],
        "vento_y": ["mean"],
        # Guardar a série de direções para o cálculo do desvio circular posterior
        "dir_vento": [lambda x: x.dropna().tolist()],
    }

    # Se for uma estação individual, capturamos a primeira instância mantendo a chave original lat, lon
    if incluir_coordenadas:
        agregacoes["lat"] = ["first"]
        agregacoes["lon"] = ["first"]

    # Agrupamento por Ano e Estação do Ano
    res = dataframe.groupby(["Ano", "Estacao_Ano"]).agg(agregacoes)

    # Achatar o MultiIndex mantendo um mapeamento limpo
    # Ex: ('temp', 'mean') vira 'temp_mean', ('lat', 'first') vira 'lat'
    novas_colunas = []
    for col, func in res.columns:
        if col in ["lat", "lon"]:
            novas_colunas.append(col)  # Mantém exatamente 'lat' e 'lon'
        else:
            novas_colunas.append(f"{col}_{func}")

    res.columns = novas_colunas
    res = res.reset_index()

    # --- 4. Cálculo do Vetor Médio e Desvio Padrão Circular ---
    direcoes_medias = []
    desvios_circulares = []

    for idx, row in res.iterrows():
        mx, my = row["vento_x_mean"], row["vento_y_mean"]

        # Direção resultante através do arcotangente (convertido para graus 0-360)
        dir_media = np.rad2deg(np.arctan2(my, mx)) % 360
        direcoes_medias.append(dir_media if not np.isnan(dir_media) else np.nan)

        # Desvio Padrão Circular
        angulos = np.deg2rad(row["dir_vento_<lambda>"])
        angulos = angulos[~np.isnan(angulos)]  # Remove nulos

        if len(angulos) > 0:
            S = np.sin(angulos).mean()
            C = np.cos(angulos).mean()
            R = np.sqrt(C**2 + S**2)

            # Limitando R a 1 para evitar imprecisão de ponto flutuante fora do domínio do logaritmo
            R = min(1.0, R)
            std_circ = np.rad2deg(np.sqrt(-2 * np.log(R)))
            desvios_circulares.append(std_circ)
        else:
            desvios_circulares.append(np.nan)

    # Inserir os resultados calculados
    res["dir_vento_vetor_medio"] = direcoes_medias
    res["dir_vento_std_circular"] = desvios_circulares

    # Seleção estruturada das colunas finais que vão para o CSV
    colunas_finais = [
        "Estacao_Ano",
        "Ano",
        "temp_mean",
        "temp_std",
        "dir_vento_vetor_medio",
        "dir_vento_std_circular",
    ]

    if incluir_coordenadas:
        colunas_finais.extend(["lat", "lon"])

    return res[colunas_finais]


# --- 5. Processamento e Salvamento dos Arquivos ---

# Grupo 1: Todas as estações juntas (Sem lat/lon)
df_todas = processar_grupo(df, incluir_coordenadas=False)
df_todas.to_csv(
    os.path.join(diretorio_saida, "todas_estacoes.csv"), index=False
)
print("✅ Arquivo 'todas_estacoes.csv' gerado com sucesso!")

# Grupos 2 a 9: Estações individuais (Com lat/lon originais)
estacoes_unicas = df["estação"].unique()

for estacao in estacoes_unicas:
    df_filtrado = df[df["estação"] == estacao]

    df_resultado = processar_grupo(df_filtrado, incluir_coordenadas=True)

    nome_arquivo = f"estacao_{str(estacao).strip().lower()}.csv"
    df_resultado.to_csv(os.path.join(diretorio_saida, nome_arquivo), index=False)
    print(f"✅ Arquivo '{nome_arquivo}' gerado com as chaves 'lat' e 'lon'!")

✅ Arquivo 'todas_estacoes.csv' gerado com sucesso!
✅ Arquivo 'estacao_ca.csv' gerado com as chaves 'lat' e 'lon'!
✅ Arquivo 'estacao_av.csv' gerado com as chaves 'lat' e 'lon'!
✅ Arquivo 'estacao_sc.csv' gerado com as chaves 'lat' e 'lon'!
✅ Arquivo 'estacao_sp.csv' gerado com as chaves 'lat' e 'lon'!
✅ Arquivo 'estacao_bg.csv' gerado com as chaves 'lat' e 'lon'!
✅ Arquivo 'estacao_cg.csv' gerado com as chaves 'lat' e 'lon'!
✅ Arquivo 'estacao_ir.csv' gerado com as chaves 'lat' e 'lon'!
✅ Arquivo 'estacao_pg.csv' gerado com as chaves 'lat' e 'lon'!
